# Customer Churn Prediction — Easy ML Pipeline

Only essential code is included: preprocessing, feature engineering, ML pipelines, hyperparameter tuning, evaluation, saving, and prediction. Every code line has a simple `#` explanation. Visualization and EDA are excluded.

## 1. Import libraries

In [ ]:
# Import pandas to read and manage the customer data.
import pandas as pd
# Import train_test_split to keep some data for testing.
# Import GridSearchCV to try different model settings.
# Import StratifiedKFold to keep churn proportions similar in each fold.
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
# Import ColumnTransformer to apply different steps to different column types.
from sklearn.compose import ColumnTransformer
# Import Pipeline to connect steps in the correct order.
from sklearn.pipeline import Pipeline
# Import tools for encoding categories, scaling numbers, and creating extra numeric features.
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
# Import SimpleImputer to fill missing values.
from sklearn.impute import SimpleImputer
# Import VarianceThreshold to remove columns that have no variation.
from sklearn.feature_selection import VarianceThreshold
# Import the Decision Tree algorithm.
from sklearn.tree import DecisionTreeClassifier
# Import the Random Forest algorithm.
from sklearn.ensemble import RandomForestClassifier
# Import metrics to measure model performance.
from sklearn.metrics import accuracy_score, classification_report
# Import SMOTE to balance classes inside the training pipeline.
from imblearn.over_sampling import SMOTE
# Import a pipeline that supports SMOTE between preprocessing and the model.
from imblearn.pipeline import Pipeline as ImbPipeline
# Import pickle to save the complete trained pipeline.
import pickle

## 2. Load and clean the data

In [ ]:
# Read the customer churn CSV file into a DataFrame.
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Remove customerID because it is an identifier, not a useful prediction feature.
df = df.drop(columns=["customerID"])

# Convert TotalCharges to numbers; blank or invalid values become missing values.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Convert the target labels: No becomes 0 and Yes becomes 1.
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# Store all input columns separately from the answer we want to predict.
X = df.drop(columns=["Churn"])
# Store Churn as the target variable.
y = df["Churn"]

## 3. Identify column types

In [ ]:
# Find numerical input columns, such as tenure and MonthlyCharges.
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
# Find category/text columns, such as Contract and InternetService.
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

## 4. Preprocessing and feature engineering with ColumnTransformer

- Numeric columns: fill missing values, create degree-2 features, then scale them.
- Categorical columns: fill missing values, then one-hot encode them.
- `ColumnTransformer` sends each column group through the correct steps.
- Feature selection and SMOTE are added after the transformer in the ML pipeline.

In [ ]:
# Start the processing steps for numerical columns.
numeric_pipeline = Pipeline(steps=[
    # Fill missing numeric values using each column's median.
    ("imputer", SimpleImputer(strategy="median")),
    # Create degree-2 numeric features, including interactions between numeric columns.
    ("feature_engineering", PolynomialFeatures(degree=2, include_bias=False)),
    # Scale numeric features so they have comparable ranges.
    ("scaler", StandardScaler())
])

# Start the processing steps for categorical/text columns.
categorical_pipeline = Pipeline(steps=[
    # Fill missing categories using the most common value in each column.
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # Turn categories into numeric indicator columns; ignore new categories at prediction time.
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Apply the numerical pipeline to numeric columns and categorical pipeline to category columns.
preprocessor = ColumnTransformer(transformers=[
    # Select numeric columns and run their steps in order.
    ("numeric", numeric_pipeline, numeric_features),
    # Select categorical columns and run their steps in order.
    ("categorical", categorical_pipeline, categorical_features)
])

## 5. Split the data

In [ ]:
# Split the data into training data and test data.
# 80% is used for training and 20% for testing.
# random_state makes the split repeatable; stratify keeps the churn ratio similar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 6. Build ML pipelines and compare algorithms

The pipeline keeps preprocessing, feature selection, SMOTE, and the model together. This helps prevent data leakage.

In [ ]:
# Create two algorithms to compare.
models = {
    # Create a Decision Tree model with a repeatable random state.
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    # Create a Random Forest model with a repeatable random state.
    "Random Forest": RandomForestClassifier(random_state=42)
}

# Run the same ML flow for each algorithm.
for name, algorithm in models.items():
    # Join preprocessing, feature selection, class balancing, and the algorithm in one pipeline.
    model_pipeline = ImbPipeline(steps=[
        # Fill missing values, engineer features, scale numbers, and encode categories.
        ("preprocessor", preprocessor),
        # Remove transformed columns that have no variation.
        ("feature_selection", VarianceThreshold()),
        # Balance the training data; the pipeline applies this only during fitting.
        ("smote", SMOTE(random_state=42)),
        # Fit the selected algorithm after the previous steps.
        ("model", algorithm)
    ])
    # Learn every pipeline step using the training data.
    model_pipeline.fit(X_train, y_train)
    # Predict churn for the untouched test data.
    predictions = model_pipeline.predict(X_test)
    # Print the algorithm name.
    print(name)
    # Print the fraction of correct predictions.
    print("Accuracy:", accuracy_score(y_test, predictions))
    # Print precision, recall, and F1-score for each class.
    print(classification_report(y_test, predictions))

## 7. Hyperparameter tuning with GridSearchCV

GridSearchCV tests different Random Forest settings using the complete pipeline and 5-fold cross-validation.

In [ ]:
# Build one complete pipeline for hyperparameter tuning.
pipeline = ImbPipeline(steps=[
    # Perform all preprocessing and feature engineering inside each training fold.
    ("preprocessor", preprocessor),
    # Remove features that do not vary.
    ("feature_selection", VarianceThreshold()),
    # Balance only the training fold to avoid data leakage.
    ("smote", SMOTE(random_state=42)),
    # Use Random Forest as the model to tune.
    ("model", RandomForestClassifier(random_state=42))
])

# List the Random Forest settings GridSearchCV should test.
param_grid = {
    # Try forests containing 100 or 200 trees.
    "model__n_estimators": [100, 200],
    # Try limiting tree depth to 10, 20, or no limit.
    "model__max_depth": [10, 20, None],
    # Try these minimum sample counts for splitting a node.
    "model__min_samples_split": [2, 5],
    # Try these minimum sample counts for a leaf node.
    "model__min_samples_leaf": [1, 2]
}

# Set up 5-fold cross-validation while preserving the class balance in each fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Search all parameter combinations using the full pipeline.
grid_search = GridSearchCV(
    # The preprocessing and model pipeline to tune.
    estimator=pipeline,
    # The parameter combinations to try.
    param_grid=param_grid,
    # The cross-validation plan.
    cv=cv,
    # Choose the settings with the best mean accuracy.
    scoring="accuracy",
    # Use all available CPU cores where possible.
    n_jobs=-1,
    # Show progress while searching.
    verbose=1
)

# Train and compare all parameter combinations using training data only.
grid_search.fit(X_train, y_train)
# Print the best settings found by the search.
print("Best parameters:", grid_search.best_params_)
# Print the best average validation accuracy found during cross-validation.
print("Best cross-validation accuracy:", grid_search.best_score_)

## 8. Evaluate the best pipeline

In [ ]:
# Keep the complete pipeline with the best hyperparameters.
best_model = grid_search.best_estimator_
# Predict churn for the test data using the same saved preprocessing steps.
y_pred = best_model.predict(X_test)
# Print accuracy on the test data that was not used for tuning.
print("Test accuracy:", accuracy_score(y_test, y_pred))
# Print precision, recall, and F1-score on the test data.
print(classification_report(y_test, y_pred))

## 9. Save the complete pipeline

In [ ]:
# Open a file in write-binary mode to save the trained pipeline.
with open("customer_churn_pipeline.pkl", "wb") as file:
    # Save preprocessing, feature engineering, feature selection, and the trained model together.
    pickle.dump(best_model, file)

## 10. Predict for a new customer

Pass values in the original column format. The saved pipeline automatically applies imputation, feature engineering, encoding, feature selection, and the trained model.

In [ ]:
# Take one test customer in the original, unprocessed column format.
new_customer = X_test.iloc[[0]]
# Predict the customer outcome; the pipeline performs all preprocessing automatically.
prediction = best_model.predict(new_customer)[0]
# Print the prediction in readable form: 1 means churn, 0 means no churn.
print("Churn" if prediction == 1 else "No Churn")